# SRPT, SJF, SPJF: M/G/1 size-based scheduling

This notebook follows **Mitzenmacher & Shahout (2025)**, *Queueing, Predictions, and LLMs* (Stochastic Systems): joint model $(X,Y)$, policies **SPJF / PSPJF / SPRPT**, and Table 3.3 ($S \sim \mathrm{Exp}(1)$, $Y|X=x \sim \mathrm{Exp}(1/x)$).

We compare **theory** (`most_queue.theory.srpt`) with **simulation** (`SizeBasedQsSim`).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from most_queue.random.distributions import ExpDistribution
from most_queue.sim.base import QsSim
from most_queue.sim.size_based import SizeBasedQsSim
from most_queue.sim.utils.predictor import ExpNoiseSimPredictor
from most_queue.theory.fifo.mg1 import MG1Calc
from most_queue.theory.srpt import MG1PsjfCalc, MG1SjfCalc, MG1SpjfCalc, MG1SrptCalc
from most_queue.theory.srpt.utils.predictor import ExpNoisePredictor, PerfectPredictor

RNG = np.random.default_rng(20260426)
EXP_RATE = 1.0  # mean service 1

PAPER_ET = {
    0.5: {"FCFS": 2.000, "SJF": 1.713, "SPJF": 1.795, "PSJF": 1.531, "PSPJF": 1.664, "SRPT": 1.425, "SPRPT": 1.653},
    0.8: {"FCFS": 5.000, "SJF": 2.882, "SPJF": 3.376, "PSJF": 2.659, "PSPJF": 3.194, "SRPT": 2.353, "SPRPT": 3.117},
    0.9: {"FCFS": 10.000, "SJF": 4.462, "SPJF": 5.527, "PSJF": 4.130, "PSPJF": 5.285, "SRPT": 3.642, "SPRPT": 5.131},
}


def sim_seed(rho, disc):
    salt = {"FCFS": 1, "SJF": 2, "SPJF": 3, "PSJF": 4, "PSPJF": 5, "SRPT": 6, "SPRPT": 7}[disc]
    return 50_000_000 + int(rho * 1_000_000) + salt


def run_sim_et(disc, rho, n_jobs):
    lam = rho
    sim = SizeBasedQsSim(1, discipline=disc, verbose=False)
    sim.generator = np.random.default_rng(sim_seed(rho, disc))
    sim.set_servers(EXP_RATE, "M")
    sim.set_sources(lam, "M")
    if disc in ("SPJF", "PSPJF", "SPRPT"):
        sim.set_predictor(ExpNoiseSimPredictor())
    return float(sim.run(n_jobs).v[0])


## 1. M/M/1 FCFS baseline
Exact $E[T] = 1/(1-\rho)$ vs `MG1Calc` vs `QsSim` (small run).


In [ ]:
rho = 0.7
lam = rho
mom = ExpDistribution.calc_theory_moments(EXP_RATE, 5)
exact = 1.0 / (1.0 - rho)

fc = MG1Calc()
fc.set_sources(lam)
fc.set_servers(mom)
et_calc = fc.run().v[0]

qs = QsSim(1, verbose=False)
qs.generator = RNG
qs.set_sources(lam, "M")
qs.set_servers(EXP_RATE, "M")
et_sim = qs.run(50_000).v[0]

print("exact E[T]", exact)
print("MG1Calc ", et_calc)
print("QsSim    ", et_sim)


## 2. SRPT theory (Schrage--Miller)
`MG1SrptCalc` integrates the conditional mean sojourn over $f(x)$.


In [ ]:
rho = 0.5
lam = rho
srpt = MG1SrptCalc()
srpt.set_sources(lam)
srpt.set_servers(EXP_RATE, "M")
et = srpt.run().v[0]
print("E[T^SRPT] at rho=0.5 (theory):", et, " paper table:", PAPER_ET[rho]["SRPT"])
for x in (0.5, 1.0, 2.0):
    print(f"  E[T^SRPT(x={x})] =", srpt.conditional_mean_response(x))


## 3. SRPT vs FCFS: theory curve
Scan $\rho$ and plot analytic $E[T]$.


In [ ]:
rhos = np.linspace(0.1, 0.95, 20)
ets_srpt, ets_fcfs = [], []
for r in rhos:
    lam = float(r)
    m = ExpDistribution.calc_theory_moments(EXP_RATE, 5)
    fc = MG1Calc()
    fc.set_sources(lam)
    fc.set_servers(m)
    ets_fcfs.append(fc.run().v[0])
    s = MG1SrptCalc()
    s.set_sources(lam)
    s.set_servers(EXP_RATE, "M")
    ets_srpt.append(s.run().v[0])

plt.figure(figsize=(7, 4))
plt.plot(rhos, ets_fcfs, label="FCFS (theory)")
plt.plot(rhos, ets_srpt, label="SRPT (theory)")
plt.xlabel("rho"); plt.ylabel("E[T]"); plt.legend(); plt.grid(True, alpha=0.3)
plt.title("M/M/1: FCFS vs SRPT (analytic)")
plt.show()


## 4. SJF vs PSJF (theory)
Both use `MG1SjfCalc` / `MG1PsjfCalc`; ordering PSJF vs SJF is not universal across all distributions (here Exp(1) service).


In [ ]:
for r in (0.5, 0.8, 0.9):
    lam = r
    sjf = MG1SjfCalc(); sjf.set_sources(lam); sjf.set_servers(EXP_RATE, "M")
    psjf = MG1PsjfCalc(); psjf.set_sources(lam); psjf.set_servers(EXP_RATE, "M")
    ts, tp = sjf.run().v[0], psjf.run().v[0]
    print(f"rho={r}: E[T_SJF]={ts:.4f}  E[T_PSJF]={tp:.4f}")


## 5. SPJF: perfect vs noisy predictions (theory)
`PerfectPredictor` matches SJF; `ExpNoisePredictor` matches Table 3.3 noise model.


In [ ]:
rho = 0.8
lam = rho
sjf = MG1SjfCalc(); sjf.set_sources(lam); sjf.set_servers(EXP_RATE, "M")
sp_perfect = MG1SpjfCalc(); sp_perfect.set_sources(lam); sp_perfect.set_servers(EXP_RATE, "M")
sp_perfect.set_predictor(PerfectPredictor())
sp_noise = MG1SpjfCalc(); sp_noise.set_sources(lam); sp_noise.set_servers(EXP_RATE, "M")
sp_noise.set_predictor(ExpNoisePredictor())
print("E[T] SJF", sjf.run().v[0])
print("E[T] SPJF perfect", sp_perfect.run().v[0])
print("E[T] SPJF Exp noise", sp_noise.run().v[0], " paper", PAPER_ET[rho]["SPJF"])


## 6. Reproduce Table 3.3 (subset): simulation vs paper
Uses moderate `N` for notebook speed; increase `N` for tighter match.


In [ ]:
N = 80_000
rows = []
for rho, paper in PAPER_ET.items():
    row = {"rho": rho}
    for disc in ("FCFS", "SJF", "SPJF", "PSJF", "PSPJF", "SRPT", "SPRPT"):
        sim = run_sim_et(disc, rho, N)
        row[f"{disc}_sim"] = sim
        row[f"{disc}_paper"] = paper[disc]
        row[f"{disc}_err%"] = 100 * abs(sim - paper[disc]) / paper[disc]
    rows.append(row)

df = pd.DataFrame(rows)
display(df.round(3))


## 7. Price of misprediction (PoM)
$\mathrm{PoM} = E[T^{\mathrm{SPJF}}] / E[T^{\mathrm{SRPT}}]$ (same $g$; theory here).


In [ ]:
rhos_pom = [0.5, 0.8, 0.9]
poms = []
for r in rhos_pom:
    lam = r
    sr = MG1SrptCalc(); sr.set_sources(lam); sr.set_servers(EXP_RATE, "M")
    sp = MG1SpjfCalc(); sp.set_sources(lam); sp.set_servers(EXP_RATE, "M"); sp.set_predictor(ExpNoisePredictor())
    et_s, et_p = sr.run().v[0], sp.run().v[0]
    poms.append(et_p / et_s)
plt.figure(figsize=(5, 3))
plt.bar([str(x) for x in rhos_pom], poms)
plt.ylabel("PoM (theory)"); plt.title("SPJF / SRPT with Exp(1) service, Exp(1/x) predictions")
plt.show()


## 8. Slowdown $T/X$ (SRPT simulation)
With `track_slowdown=True`, `get_slowdown()` returns one sample per completed job.


In [ ]:
rho = 0.7
lam = rho
sim = SizeBasedQsSim(1, discipline="SRPT", verbose=False, track_slowdown=True)
sim.generator = np.random.default_rng(12345)
sim.set_servers(EXP_RATE, "M")
sim.set_sources(lam, "M")
sim.run(30_000)
slow = np.array(sim.get_slowdown())
print("mean T/X:", slow.mean(), " median:", np.median(slow))
plt.figure(figsize=(6, 3))
plt.hist(slow[slow < 20], bins=50, density=True, alpha=0.7)
plt.xlabel("slowdown T/X"); plt.ylabel("density (truncated at 20)")
plt.title("SRPT slowdown (sim)")
plt.show()


## 9. References

- L. E. Schrage, L. W. Miller, *The Queue M/G/1 with the Shortest Remaining Processing Time Discipline*, Operations Research, 1966.
- M. Mitzenmacher, *Scheduling with Predictions and the Price of Misprediction*, ITCS 2020.
- M. Mitzenmacher, R. Shahout, *Queueing, Predictions, and LLMs: Challenges and Open Problems*, Stochastic Systems / arXiv:2503.07545, 2025.

See also `tests/test_mg1_predictions_table.py` and `examples/srpt_table.py`.
